# Duty Cycle Prediction Package - Sample Usage with HERE Maps

This notebook demonstrates how to use the duty cycle prediction package with real data from the AY71UCD dataset, including HERE map backgrounds for trajectory visualization.

## Table of Contents
1. [Setup and Imports](#setup)
2. [Load and Prepare Data](#data)
3. [Map Configuration](#maps)
4. [Route Visualization with Maps](#route-viz)
5. [Generate Driving Cycle](#generate)
6. [Vehicle Dynamics Calculations](#dynamics)
7. [Advanced Visualization with Maps](#visualization)
8. [Advanced Usage](#advanced)

## 1. Setup and Imports {#setup}

In [ ]:
# Standard library imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Add the src directory to the path for development
import sys
sys.path.append('../src')

# Import our package
from duty_cycle_prediction import (
    DrivingCycleGenerator,
    DEFAULT_VEHICLE_PARAMS,
    DEFAULT_DYNAMICS_PARAMS,
    HereMapTiles,
    create_simple_trajectory_plot
)
from duty_cycle_prediction.vehicle_dynamics import (
    calculate_wheel_power,
    calculate_fuel_consumption_rate,
    calculate_battery_consumption_rate
)

print("✅ All imports successful!")
print(f"📦 Package version: 0.1.0")

## 2. Load and Prepare Data {#data}

In [ ]:
# Load real data from AY71UCD dataset
data_file = '../data/AY71UCD/20250227_AY71UCD_Leg1.csv'
raw_data = pd.read_csv(data_file)

print(f"📊 Loaded data: {len(raw_data)} rows")
print(f"📅 Data columns: {list(raw_data.columns[:10])}...")

# Convert Unix timestamp to readable format for time range display
start_time_readable = datetime.fromtimestamp(raw_data['UnixTime'].min() / 1000).strftime('%Y-%m-%d %H:%M:%S')
end_time_readable = datetime.fromtimestamp(raw_data['UnixTime'].max() / 1000).strftime('%Y-%m-%d %H:%M:%S')
print(f"🕒 Time range: {start_time_readable} to {end_time_readable}")

# Display first few rows
raw_data.head()

In [ ]:
# Prepare route data for our package
def prepare_route_data(df, sample_factor=10):
    """Prepare route data from AY71UCD format to our package format."""
    
    # Sample data to reduce computational load (every 10th point)
    df_sampled = df.iloc[::sample_factor].copy().reset_index(drop=True)
    
    # Create route DataFrame with required columns
    route_df = pd.DataFrame({
        'Lat': df_sampled['Latitude'],
        'Lon': df_sampled['Longitude'],
        'MaxSpeed': 25.0,  # 90 km/h in m/s (truck speed limit)
        'BaseSpeed': 25.0,  # Base speed
        'TrafficSpeed': df_sampled['Spd_Kmph_x'] / 3.6,  # Convert km/h to m/s
    })
    
    # Add actions (simplified)
    actions = ['start'] + ['continue'] * (len(route_df) - 2) + ['arrive']
    route_df['Action'] = actions
    
    # Clean invalid values
    route_df = route_df.dropna().reset_index(drop=True)
    
    # Ensure traffic speed is reasonable
    route_df['TrafficSpeed'] = route_df['TrafficSpeed'].clip(0, 30)  # Max 30 m/s
    
    return route_df

# Prepare the route data
route_df = prepare_route_data(raw_data, sample_factor=50)  # Use every 50th point

print(f"🗺️ Route prepared: {len(route_df)} waypoints")
print(f"📍 Start: ({route_df.iloc[0]['Lat']:.6f}, {route_df.iloc[0]['Lon']:.6f})")
print(f"🏁 End: ({route_df.iloc[-1]['Lat']:.6f}, {route_df.iloc[-1]['Lon']:.6f})")

route_df.head()

## 3. Map Configuration {#maps}

In [ ]:
# Configure HERE Maps
# ⚠️ IMPORTANT: Replace with your actual HERE API key
# You can get a free API key at https://developer.here.com/
HERE_API_KEY = "YOUR_HERE_API_KEY_HERE"

# Initialize HERE map tiles
if HERE_API_KEY != "YOUR_HERE_API_KEY_HERE":
    map_tiles = HereMapTiles(HERE_API_KEY)
    print("🗺️ HERE Maps configured successfully")
    print(f"📡 Map tiles URL: {map_tiles.get_map_tiles_url()[:80]}...")
    USE_HERE_MAPS = True
else:
    print("⚠️ HERE API key not configured - will use simple plots")
    print("🔑 To use HERE maps, get an API key from https://developer.here.com/")
    print("📝 Then replace HERE_API_KEY with your actual key")
    USE_HERE_MAPS = False

## 4. Route Visualization with Maps {#route-viz}

In [ ]:
# Visualize the route with map background
if USE_HERE_MAPS:
    print("🗺️ Creating route visualization with HERE map background...")
    
    # Create trajectory plot with map background
    fig = map_tiles.plot_trajectory_with_map(
        lats=route_df['Lat'].tolist(),
        lons=route_df['Lon'].tolist(),
        speeds=route_df['TrafficSpeed'].tolist(),
        title="AY71UCD Leg1 Route with Speed Profile",
        figsize=(15, 12),
        zoom=12  # Adjust zoom level as needed
    )
    
    plt.show()
    
else:
    print("📊 Creating route visualization without map background...")
    
    # Create simple trajectory plot as fallback
    fig = create_simple_trajectory_plot(
        lats=route_df['Lat'].tolist(),
        lons=route_df['Lon'].tolist(),
        speeds=route_df['TrafficSpeed'].tolist(),
        title="AY71UCD Leg1 Route with Speed Profile",
        figsize=(12, 10)
    )
    
    plt.show()

In [ ]:
# Additional route analysis plots
plt.figure(figsize=(15, 10))

# Speed profile
plt.subplot(2, 2, 1)
plt.plot(route_df['TrafficSpeed'] * 3.6, label='Traffic Speed', linewidth=2)
plt.plot(route_df['MaxSpeed'] * 3.6, '--', label='Max Speed', alpha=0.7)
plt.xlabel('Waypoint Index')
plt.ylabel('Speed (km/h)')
plt.title('Speed Profile Along Route')
plt.legend()
plt.grid(True, alpha=0.3)

# Calculate distances between waypoints
from duty_cycle_prediction.driving_cycle_generator import DrivingCycleGenerator
distances = []
for i in range(len(route_df) - 1):
    dist = DrivingCycleGenerator.haversine_distance(
        route_df.iloc[i]['Lat'], route_df.iloc[i]['Lon'],
        route_df.iloc[i+1]['Lat'], route_df.iloc[i+1]['Lon']
    )
    distances.append(dist)

plt.subplot(2, 2, 2)
plt.plot(distances, color='orange', linewidth=2)
plt.xlabel('Segment Index')
plt.ylabel('Distance (m)')
plt.title('Segment Distances')
plt.grid(True, alpha=0.3)

# Cumulative distance
cumulative_distances = np.cumsum([0] + distances) / 1000  # km
plt.subplot(2, 2, 3)
plt.plot(cumulative_distances, route_df['TrafficSpeed'] * 3.6, color='green', linewidth=2)
plt.xlabel('Cumulative Distance (km)')
plt.ylabel('Speed (km/h)')
plt.title('Speed vs Distance')
plt.grid(True, alpha=0.3)

# Route statistics
total_distance = sum(distances) / 1000  # km
avg_speed = route_df['TrafficSpeed'].mean() * 3.6  # km/h

plt.subplot(2, 2, 4)
stats_text = f'''Route Statistics:

📏 Total Distance: {total_distance:.2f} km
📍 Waypoints: {len(route_df)}
🏃 Avg Speed: {avg_speed:.1f} km/h
🚀 Max Speed: {route_df["TrafficSpeed"].max()*3.6:.1f} km/h
📊 Speed Std: {route_df["TrafficSpeed"].std()*3.6:.1f} km/h
⏱️ Est. Duration: {total_distance/avg_speed*60:.1f} min

🌍 Coordinates:
Start: {route_df.iloc[0]['Lat']:.4f}, {route_df.iloc[0]['Lon']:.4f}
End: {route_df.iloc[-1]['Lat']:.4f}, {route_df.iloc[-1]['Lon']:.4f}'''

plt.text(0.05, 0.95, stats_text, fontsize=10, transform=plt.gca().transAxes,
         verticalalignment='top', fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8))
plt.title('Route Summary')
plt.axis('off')

plt.tight_layout()
plt.show()

## 5. Generate Driving Cycle {#generate}

In [ ]:
# Initialize the driving cycle generator
generator = DrivingCycleGenerator()

# Set start time
start_time = datetime(2025, 2, 27, 9, 0, 0)  # Match the data date

# Display default parameters
print("🚛 Default Vehicle Parameters:")
for key, value in DEFAULT_VEHICLE_PARAMS.items():
    print(f"  {key}: {value}")

print("\n⚙️ Generating driving cycle...")

In [ ]:
# Generate the driving cycle
try:
    # Use a subset of the route for faster processing
    route_subset = route_df.iloc[:30].copy()  # Use first 30 waypoints
    route_subset.iloc[-1, route_subset.columns.get_loc('Action')] = 'arrive'  # Ensure last action is 'arrive'
    
    driving_cycle = generator.create_driving_cycle(
        route_df=route_subset,
        start_time=start_time,
        **DEFAULT_VEHICLE_PARAMS
    )
    
    print(f"✅ Successfully generated driving cycle!")
    print(f"📊 Time steps: {len(driving_cycle)}")
    
    if len(driving_cycle) > 1:
        print(f"⏱️ Duration: {driving_cycle['timestamp'].iloc[-1] - driving_cycle['timestamp'].iloc[0]}")
        print(f"🛣️ Total distance: {driving_cycle['distance'].iloc[-1]/1000:.2f} km")
        print(f"🏃 Average speed: {driving_cycle['speed'].mean():.2f} m/s ({driving_cycle['speed'].mean()*3.6:.1f} km/h)")
        print(f"🚀 Max speed: {driving_cycle['speed'].max():.2f} m/s ({driving_cycle['speed'].max()*3.6:.1f} km/h)")
    else:
        print("⚠️ Generated cycle has only one time step - route might be too short")
    
except Exception as e:
    print(f"❌ Error generating driving cycle: {e}")
    print("📝 This might be due to route configuration. Trying with modified parameters...")
    
    # Try with a longer route and modified parameters
    try:
        route_longer = route_df.iloc[:50].copy()  # Try with 50 waypoints
        route_longer.iloc[-1, route_longer.columns.get_loc('Action')] = 'arrive'
        
        driving_cycle = generator.create_driving_cycle(
            route_df=route_longer,
            start_time=start_time,
            dt=1.0,
            smooth_speed=False,  # Disable smoothing for debugging
            **{k: v for k, v in DEFAULT_VEHICLE_PARAMS.items() if k not in ['smooth_speed']}
        )
        print(f"✅ Generated with longer route: {len(driving_cycle)} time steps")
    except Exception as e2:
        print(f"❌ Still failed: {e2}")
        driving_cycle = None

In [ ]:
# Display driving cycle data
if 'driving_cycle' in locals() and driving_cycle is not None and len(driving_cycle) > 1:
    print("📋 Driving Cycle Data (first 10 rows):")
    print(driving_cycle.head(10))
    
    print("\n📈 Statistics:")
    print(driving_cycle[['speed', 'acc', 'distance']].describe())
else:
    print("⚠️ No valid driving cycle generated for analysis")

## 6. Vehicle Dynamics Calculations {#dynamics}

In [ ]:
if 'driving_cycle' in locals() and driving_cycle is not None and len(driving_cycle) > 1:
    # Calculate power and fuel consumption for each time step
    print("⚡ Calculating vehicle dynamics...")
    
    # Vehicle parameters
    mass_kg = DEFAULT_DYNAMICS_PARAMS['mass_kg']
    
    # Calculate wheel power for each time step
    wheel_powers = []
    fuel_rates = []
    battery_rates = []
    
    for idx, row in driving_cycle.iterrows():
        # Calculate wheel power
        power = calculate_wheel_power(
            mass_kg=mass_kg,
            gradient_degrees=0.0,  # Assume flat road for simplicity
            velocity_mps=row['speed'],
            acceleration_mps2=row['acc']
        )
        wheel_powers.append(power)
        
        # Calculate fuel consumption
        fuel_result = calculate_fuel_consumption_rate(power)
        fuel_rates.append(fuel_result['rate'])
        
        # Calculate battery consumption (for EV comparison)
        battery_result = calculate_battery_consumption_rate(power)
        battery_rates.append(battery_result['rate'])
    
    # Add results to driving cycle
    driving_cycle['wheel_power_kw'] = np.array(wheel_powers) / 1000  # Convert to kW
    driving_cycle['fuel_rate_l_hr'] = fuel_rates
    driving_cycle['battery_rate_kw'] = battery_rates
    
    print(f"✅ Calculations complete!")
    print(f"⚡ Max power: {driving_cycle['wheel_power_kw'].max():.1f} kW")
    print(f"⛽ Max fuel rate: {driving_cycle['fuel_rate_l_hr'].max():.1f} L/hr")
    print(f"🔋 Max battery rate: {driving_cycle['battery_rate_kw'].max():.1f} kW")
    
    # Calculate trip totals
    trip_duration_hours = len(driving_cycle) / 3600  # Assuming 1-second time steps
    total_fuel_consumed = driving_cycle['fuel_rate_l_hr'].mean() * trip_duration_hours
    total_energy_consumed = driving_cycle['battery_rate_kw'].mean() * trip_duration_hours
    
    print(f"\n🛣️ Trip Summary:")
    print(f"  Duration: {trip_duration_hours:.3f} hours ({len(driving_cycle)/60:.1f} minutes)")
    print(f"  Distance: {driving_cycle['distance'].iloc[-1]/1000:.2f} km")
    print(f"  Average speed: {driving_cycle['speed'].mean()*3.6:.1f} km/h")
    print(f"  Fuel consumed: {total_fuel_consumed:.2f} L")
    print(f"  Energy consumed: {total_energy_consumed:.2f} kWh")
    
    if total_fuel_consumed > 0:
        print(f"  Fuel efficiency: {driving_cycle['distance'].iloc[-1]/1000/total_fuel_consumed:.2f} km/L")
    if driving_cycle['distance'].iloc[-1] > 0:
        print(f"  Energy efficiency: {total_energy_consumed/(driving_cycle['distance'].iloc[-1]/1000):.2f} kWh/km")
        
else:
    print("⚠️ No driving cycle available for dynamics calculations")

## 7. Advanced Visualization with Maps {#visualization}

In [ ]:
if 'driving_cycle' in locals() and driving_cycle is not None and len(driving_cycle) > 5:
    print("📈 Creating comprehensive visualization...")
    
    # Create comprehensive visualization
    fig, axes = plt.subplots(3, 2, figsize=(16, 14))
    
    # Time axis (in minutes)
    time_minutes = np.arange(len(driving_cycle)) / 60
    
    # Speed profile
    axes[0, 0].plot(time_minutes, driving_cycle['speed'] * 3.6, 'b-', linewidth=2, label='Actual Speed')
    axes[0, 0].plot(time_minutes, driving_cycle['speed_desired'] * 3.6, 'r--', alpha=0.7, label='Desired Speed')
    axes[0, 0].set_xlabel('Time (minutes)')
    axes[0, 0].set_ylabel('Speed (km/h)')
    axes[0, 0].set_title('Speed Profile Over Time')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # Acceleration profile
    axes[0, 1].plot(time_minutes, driving_cycle['acc'], 'g-', linewidth=2)
    axes[0, 1].axhline(y=0, color='k', linestyle='-', alpha=0.3)
    axes[0, 1].set_xlabel('Time (minutes)')
    axes[0, 1].set_ylabel('Acceleration (m/s²)')
    axes[0, 1].set_title('Acceleration Profile')
    axes[0, 1].grid(True, alpha=0.3)
    
    # Power profile
    axes[1, 0].plot(time_minutes, driving_cycle['wheel_power_kw'], 'purple', linewidth=2)
    axes[1, 0].axhline(y=0, color='k', linestyle='-', alpha=0.3)
    axes[1, 0].set_xlabel('Time (minutes)')
    axes[1, 0].set_ylabel('Wheel Power (kW)')
    axes[1, 0].set_title('Power Profile')
    axes[1, 0].grid(True, alpha=0.3)
    
    # Fuel consumption rate
    axes[1, 1].plot(time_minutes, driving_cycle['fuel_rate_l_hr'], 'orange', linewidth=2)
    axes[1, 1].set_xlabel('Time (minutes)')
    axes[1, 1].set_ylabel('Fuel Rate (L/hr)')
    axes[1, 1].set_title('Fuel Consumption Rate')
    axes[1, 1].grid(True, alpha=0.3)
    
    # Speed vs Power scatter
    scatter = axes[2, 0].scatter(driving_cycle['speed'] * 3.6, driving_cycle['wheel_power_kw'], 
                                c=driving_cycle['acc'], cmap='RdYlBu_r', alpha=0.7, s=30)
    axes[2, 0].set_xlabel('Speed (km/h)')
    axes[2, 0].set_ylabel('Wheel Power (kW)')
    axes[2, 0].set_title('Speed vs Power (colored by acceleration)')
    axes[2, 0].grid(True, alpha=0.3)
    plt.colorbar(scatter, ax=axes[2, 0], label='Acceleration (m/s²)')
    
    # Distance profile
    axes[2, 1].plot(time_minutes, driving_cycle['distance'] / 1000, 'brown', linewidth=2)
    axes[2, 1].set_xlabel('Time (minutes)')
    axes[2, 1].set_ylabel('Cumulative Distance (km)')
    axes[2, 1].set_title('Distance Profile')
    axes[2, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

else:
    print("⚠️ Insufficient data for comprehensive visualization")

In [ ]:
# Visualize the driving cycle trajectory with map background
if 'driving_cycle' in locals() and driving_cycle is not None and len(driving_cycle) > 5:
    print("🗺️ Creating driving cycle trajectory visualization...")
    
    if USE_HERE_MAPS:
        # Plot trajectory with speed coloring and map background
        fig = map_tiles.plot_trajectory_with_map(
            lats=driving_cycle['Lat'].tolist(),
            lons=driving_cycle['Lon'].tolist(),
            speeds=driving_cycle['speed'].tolist(),
            title="Generated Driving Cycle with Speed Profile",
            figsize=(15, 12),
            zoom=14
        )
        
        # Add additional information
        plt.figtext(0.02, 0.02, 
                   f"Duration: {len(driving_cycle)/60:.1f} min | "
                   f"Distance: {driving_cycle['distance'].iloc[-1]/1000:.2f} km | "
                   f"Avg Speed: {driving_cycle['speed'].mean()*3.6:.1f} km/h",
                   fontsize=10, bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
        
    else:
        # Create simple trajectory plot
        fig = create_simple_trajectory_plot(
            lats=driving_cycle['Lat'].tolist(),
            lons=driving_cycle['Lon'].tolist(),
            speeds=driving_cycle['speed'].tolist(),
            title="Generated Driving Cycle with Speed Profile",
            figsize=(12, 10)
        )
    
    plt.show()
    
    # Create a second plot with power coloring
    print("\n⚡ Creating power consumption trajectory...")
    
    if USE_HERE_MAPS:
        fig = map_tiles.plot_trajectory_with_map(
            lats=driving_cycle['Lat'].tolist(),
            lons=driving_cycle['Lon'].tolist(),
            speeds=driving_cycle['wheel_power_kw'].tolist(),
            title="Driving Cycle with Power Consumption Profile",
            figsize=(15, 12),
            zoom=14
        )
        
        # Modify colorbar label for power
        for ax in fig.get_axes():
            for child in ax.get_children():
                if hasattr(child, 'set_label'):
                    try:
                        if 'Speed' in str(child.get_label()):
                            child.set_label('Power (kW)')
                    except:
                        pass
        
    else:
        fig = create_simple_trajectory_plot(
            lats=driving_cycle['Lat'].tolist(),
            lons=driving_cycle['Lon'].tolist(),
            speeds=driving_cycle['wheel_power_kw'].tolist(),
            title="Driving Cycle with Power Consumption Profile",
            figsize=(12, 10)
        )
    
    plt.show()
    
else:
    print("⚠️ No driving cycle data available for trajectory visualization")

## 8. Advanced Usage {#advanced}

In [ ]:
# Compare different vehicle configurations with map visualization
print("🚛 Comparing different vehicle configurations...")

configurations = {
    'Conservative': {
        'v_cruise': 20.0,  # 72 km/h
        'a_acc': 0.4,      # Gentle acceleration
        'a_dec': 0.6       # Gentle deceleration
    },
    'Aggressive': {
        'v_cruise': 25.0,  # 90 km/h
        'a_acc': 0.8,      # Strong acceleration
        'a_dec': 1.0       # Strong deceleration
    },
    'Eco': {
        'v_cruise': 22.0,  # 79 km/h
        'a_acc': 0.3,      # Very gentle acceleration
        'a_dec': 0.5       # Gentle deceleration
    }
}

results = {}
driving_cycles = {}

for config_name, params in configurations.items():
    print(f"\n🔧 Testing {config_name} configuration...")
    
    # Merge with default parameters
    config_params = DEFAULT_VEHICLE_PARAMS.copy()
    config_params.update(params)
    
    try:
        # Generate driving cycle with this configuration
        route_test = route_df.iloc[:25].copy()  # Use smaller subset for speed
        route_test.iloc[-1, route_test.columns.get_loc('Action')] = 'arrive'
        
        dc = generator.create_driving_cycle(
            route_df=route_test,
            start_time=start_time,
            **config_params
        )
        
        if len(dc) > 1:
            # Calculate basic statistics
            avg_speed = dc['speed'].mean() * 3.6  # km/h
            max_speed = dc['speed'].max() * 3.6   # km/h
            avg_acc = dc['acc'].mean()
            max_acc = dc['acc'].max()
            total_distance = dc['distance'].iloc[-1] / 1000  # km
            
            # Calculate fuel consumption
            powers = [calculate_wheel_power(
                mass_kg=DEFAULT_DYNAMICS_PARAMS['mass_kg'],
                gradient_degrees=0.0,
                velocity_mps=row['speed'],
                acceleration_mps2=row['acc']
            ) for _, row in dc.iterrows()]
            
            fuel_rates = [calculate_fuel_consumption_rate(p)['rate'] for p in powers]
            avg_fuel_rate = np.mean(fuel_rates)
            
            results[config_name] = {
                'avg_speed': avg_speed,
                'max_speed': max_speed,
                'avg_acc': avg_acc,
                'max_acc': max_acc,
                'duration': len(dc),
                'distance': total_distance,
                'avg_fuel_rate': avg_fuel_rate
            }
            
            driving_cycles[config_name] = dc
            
            print(f"  ✅ Avg speed: {avg_speed:.1f} km/h")
            print(f"  ✅ Max speed: {max_speed:.1f} km/h")
            print(f"  ✅ Duration: {len(dc)} seconds")
            print(f"  ✅ Fuel rate: {avg_fuel_rate:.1f} L/hr")
        else:
            print(f"  ⚠️ Generated cycle too short")
            results[config_name] = None
        
    except Exception as e:
        print(f"  ❌ Error: {e}")
        results[config_name] = None

In [ ]:
# Compare results and visualize
valid_results = {k: v for k, v in results.items() if v is not None}

if len(valid_results) > 1:
    comparison_df = pd.DataFrame(valid_results).T
    
    print("\n📊 Configuration Comparison:")
    print(comparison_df.round(2))
    
    # Plot comparison
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    
    # Average speed comparison
    comparison_df['avg_speed'].plot(kind='bar', ax=axes[0, 0], color='blue', alpha=0.7)
    axes[0, 0].set_title('Average Speed Comparison')
    axes[0, 0].set_ylabel('Speed (km/h)')
    axes[0, 0].tick_params(axis='x', rotation=45)
    
    # Max acceleration comparison
    comparison_df['max_acc'].plot(kind='bar', ax=axes[0, 1], color='green', alpha=0.7)
    axes[0, 1].set_title('Max Acceleration Comparison')
    axes[0, 1].set_ylabel('Acceleration (m/s²)')
    axes[0, 1].tick_params(axis='x', rotation=45)
    
    # Duration comparison
    comparison_df['duration'].plot(kind='bar', ax=axes[0, 2], color='orange', alpha=0.7)
    axes[0, 2].set_title('Trip Duration Comparison')
    axes[0, 2].set_ylabel('Duration (seconds)')
    axes[0, 2].tick_params(axis='x', rotation=45)
    
    # Fuel consumption comparison
    comparison_df['avg_fuel_rate'].plot(kind='bar', ax=axes[1, 0], color='red', alpha=0.7)
    axes[1, 0].set_title('Average Fuel Rate Comparison')
    axes[1, 0].set_ylabel('Fuel Rate (L/hr)')
    axes[1, 0].tick_params(axis='x', rotation=45)
    
    # Distance comparison
    comparison_df['distance'].plot(kind='bar', ax=axes[1, 1], color='purple', alpha=0.7)
    axes[1, 1].set_title('Distance Traveled Comparison')
    axes[1, 1].set_ylabel('Distance (km)')
    axes[1, 1].tick_params(axis='x', rotation=45)
    
    # Efficiency index (distance/fuel)
    efficiency = comparison_df['distance'] / (comparison_df['avg_fuel_rate'] * comparison_df['duration'] / 3600)
    efficiency.plot(kind='bar', ax=axes[1, 2], color='brown', alpha=0.7)
    axes[1, 2].set_title('Fuel Efficiency (km/L)')
    axes[1, 2].set_ylabel('Efficiency (km/L)')
    axes[1, 2].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()
    
    # Visualize trajectories for each configuration
    if USE_HERE_MAPS and len(driving_cycles) > 0:
        print("\n🗺️ Visualizing trajectories for different configurations...")
        
        for config_name, dc in driving_cycles.items():
            if len(dc) > 5:
                print(f"\n📈 {config_name} Configuration Trajectory:")
                fig = map_tiles.plot_trajectory_with_map(
                    lats=dc['Lat'].tolist(),
                    lons=dc['Lon'].tolist(),
                    speeds=dc['speed'].tolist(),
                    title=f"{config_name} Configuration - Speed Profile",
                    figsize=(12, 10),
                    zoom=14
                )
                plt.show()
                
else:
    print("\n⚠️ Not enough valid configurations for comparison")

## Summary

This notebook demonstrated the enhanced duty cycle prediction package with HERE Maps integration:

### ✅ **Enhanced Features**
1. **🗺️ HERE Maps Integration**: Real map backgrounds for trajectory visualization
2. **📊 Data Processing**: Converting AY71UCD data with proper coordinate handling
3. **🚛 Driving Cycle Generation**: Creating realistic speed profiles from GPS data
4. **⚡ Vehicle Dynamics**: Advanced power and fuel consumption calculations
5. **📈 Rich Visualizations**: Interactive maps with speed/power color coding
6. **🔧 Configuration Testing**: Comparing different vehicle parameters with map overlays

### 🗺️ **Map Features**
- **Background Tiles**: HERE map tiles provide context for routes
- **Speed Visualization**: Color-coded trajectories show speed variations
- **Power Analysis**: Visual representation of power consumption along routes
- **Multi-Configuration**: Compare different vehicle setups on the same map

### 🔑 **Setup Requirements**
To use HERE Maps functionality:
1. Get a free API key from [HERE Developer Portal](https://developer.here.com/)
2. Replace `HERE_API_KEY` with your actual key
3. Install required dependencies: `pip install Pillow requests`

### 📍 **Next Steps**
- **Custom Routes**: Test with different AY71UCD leg files
- **Elevation Data**: Add terrain information for more accurate calculations
- **Real-time Analysis**: Process live GPS data streams
- **Fleet Optimization**: Compare multiple vehicle configurations
- **Interactive Dashboards**: Create web-based visualization tools

The package now provides professional-grade trajectory visualization with real map context, making it easier to analyze and present driving cycle results!